# ARC Prize 2026 — ARC-AGI-3

竞赛绑定 **不要改**：`arc-prize-2026-arc-agi-3`。无网。机器：`NvidiaRtxPro6000`。

## 先看清楚：这个 notebook 有两种跑法

| 你点的按钮 | 大概耗时 | 有没有玩游戏 | 榜分 |
|---|---|---|---|
| Save and Run All | 十几秒 | **没有**。只装包、写 agent、写一份假 parquet | 不变 |
| Submit to Competition | 数小时 | **有**。网关把 hidden 游戏一帧帧塞给 MyAgent | 才会变 |

下面四格代码：1 装轮子 → 2 写出智能体 → 3 **只有评分重跑才打游戏** → 4 假提交（让提交按钮出现）。
Save and Run All 会跳过第 3 格里的 `main.py`，所以十几秒 COMPLETE 不代表得了分。


# 我们的方案：怎么玩游戏（对着官方赛题设计）

官方报告考四件事：探索、建模、自己定目标、规划执行。
至少 6 关。第 1 关是教程。后面关要组合前面学会的机制。
整局分按关卡号加权；只打完前几关会被封顶（5 关打完 3 关最多 40%）。

公开哈希图每关清空地图，等于把教程关白打了。我们不这么干。

## 一句话

教程关把「怎么赢」写进技能纸；后面关用技能纸定目标，A* 走近再按。

## 每一步

```mermaid
flowchart TD
    A[网关送来一帧] --> B[抹掉分数条]
    B --> C{技能纸已经知道赢的颜色或角色了吗}
    C -->|知道| D[A星向目标走一步]
    D --> E{这一步会踩上目标吗}
    E -->|会| F[下一步立刻 ACTION5]
    E -->|还没有| G[先走这一步]
    C -->|还不知道| H[便宜实验: 先按再走 认键位]
    H --> I[把会走/会点/会按 写进技能纸]
    I --> A
    F --> A
    G --> A
```

## 计分决定预算

教程关权重最小：把机制学到手就过。后面关权重大、要组合，多给步数。
没打完最后几关，前面再快也封顶。所以不要为了省第 1 关的步数而放弃整局。

Gemma-4 只在图穷了才问。官方：模型内部思考不计步。不按游戏名写死。


In [ ]:
# 【步骤】1/4 无网安装竞赛自带的 arc-agi 轮子（不要 pip 上网）
!pip install --no-index --find-links \
    /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels \
    arc-agi python-dotenv